# Antenatal model validation and feature importance

The current releases use age at pregnancy start, pregnancy history and multiple pregnancy. LBW also uses education.
The original modeling notebooks document historical research recipes. Run `python -m ml.src.train_antenatal_models` and `python -m ml.src.validate_country` before this notebook.
Independent prospective clinical validation and clinical decision cutoffs remain pending.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "contracts").is_dir())
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt

cohorts = ["perinatal", "perinatal_anc", "lbw"]
reports = {name: json.loads((ROOT / "ml/models/antenatal" / name / "selected.json").read_text()) for name in cohorts}
country_reports = {name: json.loads((ROOT / "ml/models/antenatal" / name / "country_validation.json").read_text()) for name in cohorts}
pd.DataFrame([{ "cohort": name, "algorithm": report["algorithm"], **report["test"] } for name, report in reports.items()])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for axis, (name, report) in zip(axes, reports.items()):
    curve = report["calibration"]
    axis.plot(curve["predicted"], curve["observed"], marker="o", label="Held-out quantile bins")
    limit = max(curve["predicted"] + curve["observed"]) * 1.1
    axis.plot([0, limit], [0, limit], linestyle="--", color="gray", label="Perfect calibration")
    axis.set(title=name, xlabel="Mean estimated probability", ylabel="Observed outcome fraction")
    axis.legend()
plt.show()


## Kenya-to-Tanzania transfer

Selection and calibration use Kenya-only grouped folds. Tanzania outcomes are
used only for the final evaluation. Survey-weighted metrics are available in
each country report alongside these unweighted results.


In [ ]:
pd.DataFrame([{ "cohort": name, "algorithm": report["algorithm"], **report["test"] } for name, report in country_reports.items()])


## Global feature importance

Permutation importance measures the change in held-out average precision when
one input is shuffled. Correlated and derived inputs can share importance.
These aggregate associations require clinical interpretation. Patient-level
causal explanations require further evaluation.


In [ ]:
import joblib
from sklearn.inspection import permutation_importance
from ml.src.dataset import OUTCOMES
from ml.src.antenatal import dataset as build_cohort
from ml.src.artifacts import validate_bundle

fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
for axis, name in zip(axes, cohorts):
    bundle = joblib.load(ROOT / "ml/models/antenatal" / name / "selected.joblib")
    model, manifest = validate_bundle(bundle, name)
    test = build_cohort(name).query("split == 'test'")
    importance = permutation_importance(
        model, test[manifest["features"]], test[OUTCOMES[name]],
        scoring="average_precision", n_repeats=5, random_state=42, n_jobs=2,
    )
    order = importance.importances_mean.argsort()
    axis.barh([manifest["features"][index] for index in order], importance.importances_mean[order])
    axis.set(title=name, xlabel="Decrease in held-out PR-AUC")
plt.show()
